In [ ]:
# @title Copyright 2026 Google LLC
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Weather forecasting and storm risk nowcasting with Google Earth Engine, BigQuery ML, and Gemini

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/python-docs-samples/blob/main/people-and-planet-ai/weather-forecasting/README.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fpython-docs-samples%2Fmain%2Fpeople-and-planet-ai%2Fweather-forecasting%2FREADME.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-enterprise-logo.png" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/python-docs-samples/blob/main/people-and-planet-ai/weather-forecasting/README.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<details>
<summary><b>Using Colab Enterprise?</b> (Click if your project has no default VPC network)</summary>

If your organization enforces `constraints/compute.skipDefaultNetworkCreation`, create a VPC network and subnet with Private Google Access before connecting to a Colab Enterprise runtime:

```bash
gcloud compute networks create colab-network --subnet-mode=custom
gcloud compute networks subnets create colab-subnet --network=colab-network --region=us-central1 --range=10.0.0.0/24 --enable-private-ip-google-access
```

</details>

---

| Author |
| --- |
| [Google Cloud](https://github.com/GoogleCloudPlatform) |

## Overview

Physics-based Numerical Weather Prediction (NWP) models solve coupled differential equations across a 3D grid, requiring supercomputers and hours of runtime. As introduced in the [People and Planet AI: Weather Forecasting video](https://www.youtube.com/watch?v=6-UJzEXMvGY), Google Research pioneered a deep learning approach that learns cloud and precipitation evolution directly from satellite and radar observations. Foundation models in BigQuery ML (TimesFM 2.5 using `AI.FORECAST`) and global ensemble models (Google DeepMind's [WeatherNext 3](https://developers.google.com/weathernext/guides/models)) let you generate probabilistic forecasts in seconds without training a custom neural network.

Because extreme hurricanes don't make landfall every day, weather engineers validate pipelines in two stages. In Steps 1 through 5, you replay Hurricane Melissa's October 2025 Jamaica landfall (a *hindcast*), holding out 6 hours of observed post-landfall rainfall to score forecast accuracy. In Step 6, you run the same pipeline on live satellite and mesoscale observations across the San Francisco and Monterey Bay Area (or your own coordinates).

### Objectives

In this tutorial, you learn how to:

* Align multi-resolution Google Earth Engine satellite (`NOAA/GOES/19/MCMIPF`, `NOAA/GOES/18/MCMIPF`, `NOAA/GOES/16/MCMIPF`), precipitation (`NASA/GPM_L3/IMERG_V07`), and elevation (`MERIT/DEM/v1_0_3`) rasters using `fetch_weather_patch.py`.
* Load a 36-hour storm time series into BigQuery using Apache Beam (`load_weather_time_series.py`).
* Run zero-shot precipitation forecasting in BigQuery ML with `AI.FORECAST` (TimesFM 2.5) and evaluate predictions against holdout ground-truth observations (`forecast_precipitation.py`).
* Generate a structured `StormRiskAssessment` report from multi-band satellite imagery and tabular forecasts using Gemini 3.8 Flash (`analyze_storm_risk.py`).
* Compare 64-member WeatherNext 3 storm ensembles and live 5-city San Francisco and Monterey Bay microclimate forecasts.

### Costs

This tutorial uses billable components of Google Cloud:

* [BigQuery](https://cloud.google.com/bigquery/pricing) (storage and `AI.FORECAST` query processing)
* [Dataflow](https://cloud.google.com/dataflow/pricing) (if you run the Apache Beam pipeline on `DataflowRunner`; this notebook defaults to local `DirectRunner`)
* [Gemini Enterprise Agent Platform](https://cloud.google.com/gemini-enterprise-agent-platform/generative-ai/pricing) (`gemini-3.8-flash` multimodal token usage)
* [Cloud Storage](https://cloud.google.com/storage/pricing) (temporary staging bucket for BigQuery file loads)

Use the [Pricing Calculator](https://cloud.google.com/products/calculator) to generate a cost estimate based on your projected usage.

### Prerequisites

* A Google Cloud project with billing enabled.
* Google Earth Engine enabled and registered on your project on the [Earth Engine configuration page in the Google Cloud console](https://console.cloud.google.com/earth-engine/configuration).
* IAM roles: BigQuery Admin (`roles/bigquery.admin`), Storage Admin (`roles/storage.admin`), and Agent Platform User (`roles/aiplatform.user`).
* Optional: [WeatherNext dataset access](https://developers.google.com/weathernext/guides/access-forecast) on your Google Account email to query live operational assets in Step 5 (Step 5 falls back automatically to pre-extracted ensemble data while access is pending).

## Get started

### Install required packages

Download the `people-and-planet-ai/weather-forecasting` sample files and install the required Python packages into the active notebook kernel (takes about 30 seconds):

In [ ]:
# @title Download sample files and install packages (~30 seconds)
from pathlib import Path
import sys
import tempfile
import urllib.request

BRANCH = "main"
SETUP_PATH = Path(tempfile.gettempdir()) / "notebook_setup.py"
for parent in (Path.cwd(), *Path.cwd().parents):
    candidate = parent / "scripts/notebook-setup/notebook_setup.py"
    if candidate.is_file():
        SETUP_PATH = candidate
        break
else:
    if not SETUP_PATH.is_file():
        url = (
            "https://raw.githubusercontent.com/GoogleCloudPlatform/"
            f"python-docs-samples/{BRANCH}/scripts/notebook-setup/notebook_setup.py"
        )
        with urllib.request.urlopen(url, timeout=30) as response:
            SETUP_PATH.write_bytes(response.read())

setup_dir = str(SETUP_PATH.parent.resolve())
if setup_dir not in sys.path:
    sys.path.insert(0, setup_dir)
import notebook_setup

notebook_setup.bootstrap("people-and-planet-ai/weather-forecasting", branch=BRANCH)

### Configure your Google Cloud project and BigQuery resources

Set your `PROJECT_ID`, `LOCATION`, `BUCKET_NAME`, `BIGQUERY_DATASET`, and `BIGQUERY_TABLE` values:

In [ ]:
import logging
import os
import re
import subprocess

logging.getLogger("google_auth_httplib2").setLevel(logging.ERROR)

# fmt: off
PROJECT_ID = "example-project-id"  # @param {type:"string", placeholder:"example-project-id", isTemplate:true}
LOCATION = "US"  # @param {type:"string"}
BUCKET_NAME = "example-weather-bucket"  # @param {type:"string"}
BIGQUERY_DATASET = "example_weather_forecasting"  # @param {type:"string"}
BIGQUERY_TABLE = "example_storm_observations"  # @param {type:"string"}
# fmt: on

if not PROJECT_ID or PROJECT_ID == "example-project-id":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT") or subprocess.check_output(
        ["gcloud", "config", "get-value", "project"], text=True
    ).strip()
LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION", LOCATION)
if not BUCKET_NAME or BUCKET_NAME == "example-weather-bucket":
    BUCKET_NAME = os.environ.get("BUCKET_NAME", f"{PROJECT_ID}-weather-forecasting")
BIGQUERY_DATASET = os.environ.get("BIGQUERY_DATASET", BIGQUERY_DATASET)
BIGQUERY_TABLE = os.environ.get("BIGQUERY_TABLE", BIGQUERY_TABLE)

assert re.fullmatch(r"[a-zA-Z0-9][a-zA-Z0-9_.:-]{0,127}", PROJECT_ID), "Invalid PROJECT_ID"
assert re.fullmatch(r"[a-zA-Z0-9][a-zA-Z0-9_.-]{1,220}", BUCKET_NAME), "Invalid BUCKET_NAME"
assert re.fullmatch(r"[a-zA-Z0-9_]{1,1024}", BIGQUERY_DATASET), "Invalid BIGQUERY_DATASET"
assert re.fullmatch(r"[a-zA-Z0-9_-]{1,1024}", BIGQUERY_TABLE), "Invalid BIGQUERY_TABLE"

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
print(f"Location:         {LOCATION}")
print(f"BigQuery Dataset: {BIGQUERY_DATASET}")
print(f"BigQuery Table:   {BIGQUERY_TABLE}")

### Authenticate your notebook environment

If you're running this notebook in Google Colab, authenticate with your Google Cloud user credentials. In Colab Enterprise, Application Default Credentials (ADC) are preconfigured:

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### Enable required Google Cloud APIs and create the staging bucket

Enable the BigQuery, Agent Platform (`aiplatform.googleapis.com`), Earth Engine, Dataflow, and Cloud Storage APIs, and create a regional Cloud Storage bucket with soft-delete turned off (`--soft-delete-duration=0s`) for temporary BigQuery staging files if the bucket doesn't already exist (takes about 30 seconds):

In [ ]:
# Takes ~30 seconds to enable Google Cloud APIs and create the staging bucket if needed.
!gcloud services enable bigquery.googleapis.com aiplatform.googleapis.com dataflow.googleapis.com earthengine.googleapis.com storage.googleapis.com --project={PROJECT_ID}

bucket_check = subprocess.run(
    ["gcloud", "storage", "buckets", "describe", f"gs://{BUCKET_NAME}", f"--project={PROJECT_ID}"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
if bucket_check.returncode != 0:
    !gcloud storage buckets create gs://{BUCKET_NAME} --project={PROJECT_ID} --location={LOCATION} --soft-delete-duration=0s

## 1. Retrieve and visualize historical storm satellite, precipitation, and terrain patches

In Steps 1 through 5, you replay Hurricane Melissa at 17:00 UTC on October 28, 2025, when its Category 5 eyewall reached southwestern Jamaica (18.1°N, 78.0°W).

Earth Engine queries satellite and precipitation archives by UTC timestamp and geographic coordinates rather than by storm name. To inspect a different historical storm in this step, pass its `iso_time`, `latitude`, `longitude`, and `storm_name` to `get_storm_patch_arrays()` (which automatically selects GOES-18 west of -105°, GOES-19 from April 7, 2025 onward, or GOES-16 for earlier Atlantic events); to run Steps 2 through 4 on another storm, load that event's hourly rainfall series into BigQuery in Step 2, or jump to Step 6 to query live weather for any coordinates.

Rather than downloading raw satellite archives, [`fetch_weather_patch.py`](fetch_weather_patch.py) calls the Earth Engine High Volume endpoint (`ee.data.computePixels`) to reproject and align three public datasets onto a shared 64x64 pixel grid:

1. **Panel 1, cloud and moisture (`NOAA/GOES/19/MCMIPF`, 2 km native resolution)**: geostationary satellites scan the Americas continuously. Earth Engine's `MCMIPF` collection provides calibrated reflectance and brightness temperature bands, which the script scales to `[0, 255]` RGB values to highlight cold cloud tops and deep water vapor.
2. **Panel 2, surface precipitation rate (`NASA/GPM_L3/IMERG_V07`, 10 km native resolution, in mm/hr)**: NASA's Global Precipitation Measurement (GPM) constellation uses microwave and radar sensors to measure falling rain beneath the cloud deck. Dark navy indicates dry sky (0 mm/hr), cyan and green indicate light-to-moderate rain (2-10 mm/hr), yellow and amber indicate heavy rain (15-25 mm/hr), and red to magenta (>30 mm/hr) indicates extreme rainfall.
3. **Panel 3, terrain elevation (`MERIT/DEM/v1_0_3`, resampled from 90 m DEM, in meters)**: steep mountain ridges such as Jamaica's Blue Mountains (2,256 m) force moist winds upward (*orographic lift*), intensifying windward rainfall and runoff.
4. **Panel 4, rainbands overlaid on terrain**: alpha-blends the translucent GOES cloud layer and GPM rainbands (>= 1.2 mm/hr) onto the MERIT DEM topography to highlight where heavy rainbands intersect steep terrain.

Fetch the aligned Earth Engine rasters and render the 4-panel storm overview (takes about 15 seconds):

In [ ]:
# Takes ~15 seconds to fetch and align the multi-resolution Earth Engine rasters.
import fetch_weather_patch
import visualize

patch_data = fetch_weather_patch.get_storm_patch_arrays(
    project_id=PROJECT_ID,
    iso_time="2025-10-28T17:00:00",
    latitude=18.1,
    longitude=-78.0,
    storm_name="Hurricane Melissa (Category 5 Jamaica Landfall)",
    patch_size=64,
)
print(f"Storm:       {patch_data['storm_name']}")
print(f"Timestamp:   {patch_data['iso_time']}")
print(f"Satellite:   {patch_data['goes_satellite']}")
print(f"Data Source: {patch_data['source']}")

overview_fig = visualize.create_storm_overview_figure(patch_data)
overview_fig.show()

Export the four aligned panels (GOES cloud and moisture, GPM rainfall, MERIT DEM elevation, and the rainband-on-terrain overlay) as a single PNG strip (`storm_multiband_strip.png`) for multimodal analysis in Step 4. Including the three unoccluded sensor panels alongside the blended overlay lets Gemini 3.8 Flash inspect exact rainfall and elevation colors directly:

In [ ]:
MULTIBAND_STRIP_PATH = "storm_multiband_strip.png"
png_bytes = visualize.export_multiband_strip_png_bytes(patch_data)
with open(MULTIBAND_STRIP_PATH, "wb") as f:
    f.write(png_bytes)
print(f"Saved 4-panel multi-band strip to {MULTIBAND_STRIP_PATH} ({len(png_bytes):,} bytes) for Gemini 3.8 Flash.")

## 2. Load 36 hours of historical storm observations into BigQuery with Apache Beam

A single satellite snapshot at 17:00 UTC can't show how fast a storm is moving or whether rainfall is intensifying. A forecasting model needs multiple historical time steps to capture storm momentum. In this step, you load 36 hourly Hurricane Melissa observations leading up to landfall (06:00 UTC on October 27 through 17:00 UTC on October 28, 2025).

Because most locations and hours have 0 mm/hr of rain, custom classification models often group rainfall into discrete intensity buckets so dry hours don't dominate training. Foundation models such as TimesFM 2.5 instead forecast continuous precipitation rates (`precipitation_mm_hr`) directly from the raw time series without manual class bucketing.

Run [`load_weather_time_series.py`](load_weather_time_series.py) using Apache Beam `DirectRunner` to populate the BigQuery observation table (takes 15-25 seconds):

In [ ]:
# Takes ~15-25 seconds to load 36 hourly storm observations into BigQuery via Apache Beam.
!python load_weather_time_series.py \
  --project={PROJECT_ID} \
  --dataset={BIGQUERY_DATASET} \
  --table={BIGQUERY_TABLE} \
  --runner=DirectRunner \
  --temp_location=gs://{BUCKET_NAME}/temp

## 3. Hindcast the next 6 hours of the storm with BigQuery ML `AI.FORECAST`

Using the 36 hours of historical storm inputs up to 17:00 UTC on October 28, 2025, forecast the next 6 hours of Hurricane Melissa (18:00 UTC to 23:00 UTC) with BigQuery ML's zero-shot `AI.FORECAST` function and Google Research's TimesFM 2.5 foundation model.

Because physical precipitation can't be negative (< 0 mm/hr), the script clamps statistical lower prediction bounds at zero (`max(0.0, lower_bound_mm_hr)`), matching the non-negative ReLU output constraint used in neural weather models.

Run [`forecast_precipitation.py`](forecast_precipitation.py) to generate the 6-hour forecast and 90% prediction intervals (takes about 15 seconds):

In [ ]:
# Takes ~15 seconds to run zero-shot TimesFM 2.5 forecasting in BigQuery ML.
!python forecast_precipitation.py {PROJECT_ID} {BIGQUERY_DATASET} {BIGQUERY_TABLE}

### Evaluate the 6-hour hindcast against ground-truth observations

Because this is a historical storm hindcast, you can compare the forecast against what actually occurred from 18:00 to 23:00 UTC on October 28, 2025. Passing `output_historical_time_series => TRUE` to `AI.FORECAST` returns both the 36-hour input history (`time_series_type = 'history'`) and the 6-hour forecast (`time_series_type = 'forecast'`) in a single BigQuery ML query:

* **Blue solid line (36-hour inputs)**: historical rainfall observations passed to `AI.FORECAST` (06:00 UTC on October 27 through 17:00 UTC on October 28).
* **Green solid line with diamonds (6-hour holdout labels)**: post-landfall rainfall (18:00 to 23:00 UTC on October 28) held back from the model.
* **Red dashed line and shaded band (TimesFM 2.5 hindcast and 90% interval)**: zero-shot predictions scored with Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and 90% interval coverage:

In [ ]:
from google.cloud import bigquery

bq_client = bigquery.Client(project=PROJECT_ID)
table_ref = f"{PROJECT_ID}.{BIGQUERY_DATASET}.{BIGQUERY_TABLE}"
fc_sql = f"""
    SELECT time_series_timestamp, time_series_type, time_series_data,
           prediction_interval_lower_bound AS lower_bound_mm_hr,
           prediction_interval_upper_bound AS upper_bound_mm_hr
    FROM AI.FORECAST(
      TABLE `{table_ref}`, data_col => 'precipitation_mm_hr',
      timestamp_col => 'observation_time', model => 'TimesFM 2.5',
      id_cols => ['storm_name'], horizon => 6, confidence_level => 0.90,
      output_historical_time_series => TRUE
    ) ORDER BY time_series_timestamp ASC
"""
all_rows = [dict(r) for r in bq_client.query(fc_sql).result()]
historical_rows = [{"observation_time": r["time_series_timestamp"], "precipitation_mm_hr": r["time_series_data"]} for r in all_rows if r["time_series_type"] == "history"]
forecast_rows = [{"forecast_timestamp": r["time_series_timestamp"], "forecasted_precipitation_mm_hr": r["time_series_data"], "lower_bound_mm_hr": r["lower_bound_mm_hr"], "upper_bound_mm_hr": r["upper_bound_mm_hr"]} for r in all_rows if r["time_series_type"] == "forecast"]
eval_metrics = visualize.evaluate_forecast_against_ground_truth(forecast_rows)
print(
    f"Holdout Evaluation (6h Unseen Ground Truth) -> MAE: {eval_metrics['mae_mm_hr']:.2f} mm/hr | "
    f"RMSE: {eval_metrics['rmse_mm_hr']:.2f} mm/hr | 90% Coverage: {eval_metrics['interval_coverage_pct']:.0f}%"
)
forecast_fig = visualize.create_forecast_comparison_figure(
    historical_rows, forecast_rows, "Hurricane Melissa (Oct 27-28, 2025)", eval_metrics["holdout_rows"]
)
forecast_fig.show()

## 4. Synthesize a multimodal flood-risk report with Gemini 3.8 Flash

Flood risk depends on both rainfall intensity and local topography. Thirty millimeters of rain per hour over open ocean is benign, whereas the same rainfall over saturated 2,000 m mountain slopes funnels into narrow river valleys and triggers flash floods.

Run [`analyze_storm_risk.py`](analyze_storm_risk.py) to pass both the 4-panel satellite strip (`storm_multiband_strip.png`) from Step 1 and the 6-hour `AI.FORECAST` trajectory from Step 3 to Gemini 3.8 Flash (`gemini-3.8-flash` on the `global` endpoint) with a Pydantic `StormRiskAssessment` response schema (takes about 15 seconds):

* **`Display Name` (`flood_risk_level`)**: overall flood severity classification for the landfall zone.
* **`Orographic Amplification Risk`**: how the mountain ridges in Panels 3 and 4 interact with the rainbands in Panel 2.
* **`Storm Summary` and recommended actions**: prioritized emergency response steps:

In [ ]:
# Takes ~15 seconds to synthesize the multimodal flood-risk assessment with Gemini 3.8 Flash.
import numpy as np

_trajectory = ", ".join(
    f"+{idx}h: {float(r['forecasted_precipitation_mm_hr']):.1f} mm/hr"
    for idx, r in enumerate(forecast_rows, start=1)
)
os.environ["WEATHER_FORECAST_CONTEXT"] = (
    f"Storm: {patch_data['storm_name']} ({patch_data['iso_time']}). "
    f"Observed peak GPM IMERG V07 rainfall: {float(np.max(patch_data['precipitation'])):.1f} mm/hr "
    f"over MERIT DEM terrain (max elevation {float(np.max(patch_data['elevation'])):.0f} m). "
    f"BigQuery AI.FORECAST (TimesFM 2.5) next {len(forecast_rows)} hours trajectory: {_trajectory}."
)
!python analyze_storm_risk.py {PROJECT_ID} {MULTIBAND_STRIP_PATH}

## 5. Compare a 48-hour, 64-member storm ensemble with WeatherNext 3

While `AI.FORECAST` provides a fast 6-hour point nowcast from station or grid history, forecasting a hurricane 2 to 5 days out requires modeling atmospheric uncertainty across the globe.

Google DeepMind's [WeatherNext 3](https://developers.google.com/weathernext/guides/models) runs 64 slightly perturbed 3D atmospheric simulations (*ensemble members*) every hour. Unlike custom inference workflows that require provisioning NVIDIA A100 or H100 GPU quota, Google publishes operational WeatherNext 3 forecasts every hour across [Earth Engine](https://developers.google.com/weathernext/guides/earth-engine), BigQuery, and Cloud Storage (Zarr v3). In the [WeatherNext 3 (0.1°) Earth Engine catalog dataset](https://developers.google.com/earth-engine/datasets/catalog/projects_gcp-public-data-weathernext_assets_weathernext_3_0_0_0p1deg) (`projects/gcp-public-data-weathernext/assets/weathernext_3_0_0_0p1deg`), all 64 members are pre-summarized into percentile bands across 114 variables:

* **`p50` (ensemble median, dark blue line)**: middle scenario where half of the 64 members predict less rain and half predict more.
* **`p25` to `p75` (interquartile range, darker blue band)**: middle 50% of ensemble members.
* **`p10` to `p90` (10th to 90th percentile, light blue band and orange dotted line)**: outer 80% spread. When the `p90` tail-risk curve spikes well above `p50`, a subset of storm tracks produces extreme rainfall that requires early flood preparedness.

### Enable live WeatherNext 3 dataset access

To query live operational WeatherNext 3 assets directly, submit your Google Cloud account email on the [WeatherNext Data Request form](https://developers.google.com/weathernext/guides/access-forecast) (requests are typically approved within 5-7 business days, and a single approval unlocks Earth Engine, BigQuery, and Cloud Storage). If your account isn't allowlisted yet, `get_weathernext_ensemble_forecast()` falls back automatically to a pre-extracted ensemble trajectory so you can run the following cell without interruption.

> **Note**: WeatherNext 3 forecasts are experimental data and aren't official severe weather warnings. Always defer to official alerts from local emergency authorities. For more information, see the [WeatherNext disclaimers](https://developers.google.com/weathernext/guides/disclaimers).

Query the `weathernext_3_0_0_0p1deg` catalog in Earth Engine and render the 48-hour ensemble plume chart (takes about 15 seconds):

In [ ]:
# Takes ~15 seconds to query the WeatherNext 3 ensemble catalog and render the plume chart.
wn_data = fetch_weather_patch.get_weathernext_ensemble_forecast(
    project_id=PROJECT_ID,
    region_name="Hurricane Melissa (Jamaica Landfall)",
)
print(f"Collection ID:        {wn_data['collection_id']}")
print(f"Latest Live Asset ID: {wn_data['latest_asset_id']}")
print(f"Latest Init Time:     {wn_data['latest_init_utc']}")
print(f"Total Ensemble Bands: {wn_data['total_bands']}")
print(f"Precip Percentiles:   {', '.join(wn_data['precip_bands'])}")
for r in wn_data["ensemble_rows"]:
    print(
        f"  +{r['lead_hours']:02d}h ({r['valid_time_utc']}): "
        f"p50={r['p50_mm_hr']:4.1f} mm/hr | mean={r['mean_mm_hr']:4.1f} mm/hr | "
        f"p25-p75=[{r['p25_mm_hr']:4.1f}, {r['p75_mm_hr']:4.1f}] | "
        f"p10-p90=[{r['p10_mm_hr']:4.1f}, {r['p90_mm_hr']:4.1f}]"
    )

wn_fig = visualize.create_weathernext_ensemble_figure(wn_data)
wn_fig.show()

## 6. Forecast live weather and microclimates across the San Francisco and Monterey Bay Area

After validating the workflow on Hurricane Melissa, switch to live weather observations across Northern and Central California. Cold Pacific upwelling, coastal marine gaps, and 850 m mountain ridges create distinct microclimates across five neighboring cities:

1. **San Francisco (37.77°N, 122.42°W, Golden Gate Marine Gap)**: exposed to cold Pacific air through the Golden Gate; cool, breezy, and quick to draw in coastal fog.
2. **Palo Alto (37.44°N, 122.14°W, Mid-Peninsula Rain Shadow)**: sheltered east of the 750 m Santa Cruz Mountains crest, blocking much of the marine fog and coastal rain.
3. **San Jose (37.34°N, 121.89°W, Inland Santa Clara Valley)**: sheltered between the Santa Cruz Mountains and the 1,280 m Diablo Range, producing warmer and drier conditions.
4. **Santa Cruz (36.97°N, 122.03°W, Windward Mountain Slopes)**: faces Monterey Bay backed by steep coastal mountains that lift moist ocean breezes upward.
5. **Monterey (36.60°N, 121.89°W, Upwelling Bay Peninsula)**: bordered by cold waters from the Monterey Submarine Canyon, maintaining cool temperatures and high humidity.

Query the latest GOES-18 West scan (`NOAA/GOES/18/MCMIPF`), 2.5 km NOAA RTMA surface analysis (`NOAA/NWS/RTMA`), 72-hour NOAA GFS forecast (`NOAA/GFS0P25`), and MERIT DEM terrain across all five cities, or customize `LOCAL_REGION_NAME`, `LOCAL_LATITUDE`, and `LOCAL_LONGITUDE` for another location (takes about 15 seconds):

In [ ]:
# Takes ~15 seconds to pull live GOES-18 West, 2.5km NOAA RTMA, and 72h GFS microclimate data.
# fmt: off
LOCAL_REGION_NAME = "San Francisco & Monterey Bay Area, CA"  # @param {type:"string"}
LOCAL_LATITUDE = 37.7749  # @param {type:"number"}
LOCAL_LONGITUDE = -122.4194  # @param {type:"number"}
# fmt: on
LOCAL_REGION_NAME = os.environ.get("LOCAL_REGION_NAME", LOCAL_REGION_NAME)

local_result = fetch_weather_patch.get_local_weather_forecast(
    project_id=PROJECT_ID,
    region_name=LOCAL_REGION_NAME,
    latitude=float(LOCAL_LATITUDE),
    longitude=float(LOCAL_LONGITUDE),
)
local_patch = local_result["patch_data"]
print(f"Region:      {local_patch['storm_name']}")
print(f"Data Source: {local_patch['source']}\n")
if local_result["microclimate_rows"]:
    print("5-City Bay Area & Monterey Bay Microclimate Comparison (Live 2.5km RTMA + 24h GFS):")
    for m in local_result["microclimate_rows"]:
        print(
            f"  {m['city']:14s} ({m['microclimate_zone']:25s}) | Elev {m['elevation_m']:4.0f}m | "
            f"Now: {m['current_temp_c']:4.1f}°C ({m['current_temp_f']:4.1f}°F), wind {m['current_wind_ms']:3.1f} m/s | "
            f"+24h: {m['forecast_24h_temp_c']:4.1f}°C ({m['forecast_24h_temp_f']:4.1f}°F), RH {m['forecast_24h_rh_pct']:2.0f}%"
        )
    micro_fig = visualize.create_bay_area_microclimate_figure(local_result["microclimate_rows"])
    micro_fig.show()
local_fig = visualize.create_storm_overview_figure(local_patch)
local_fig.show()

### Synthesize a live regional weather briefing with Gemini 3.8 Flash

Export the 4-panel satellite and terrain strip (`local_weather_multiband_strip.png`) and pass it with the `NOAA/GFS0P25` 72-hour trajectory (and 5-city Bay Area microclimate comparison when querying the default Bay Area corridor) into [`analyze_storm_risk.py`](analyze_storm_risk.py) (takes about 15 seconds):

In [ ]:
# Takes ~15 seconds to generate the localized weather briefing with Gemini 3.8 Flash.
LOCAL_STRIP_PATH = "local_weather_multiband_strip.png"
local_png_bytes = visualize.export_multiband_strip_png_bytes(local_patch)
with open(LOCAL_STRIP_PATH, "wb") as f:
    f.write(local_png_bytes)
print(f"Saved 4-panel multi-band strip to {LOCAL_STRIP_PATH} ({len(local_png_bytes):,} bytes) for Gemini 3.8 Flash.")

os.environ["WEATHER_FORECAST_CONTEXT"] = local_result["forecast_context"]
!python analyze_storm_risk.py {PROJECT_ID} {LOCAL_STRIP_PATH}

### Validate target state

Verify that the historical hurricane rasters, 36-hour BigQuery observations, 6-hour `AI.FORECAST` hindcast, holdout evaluation metrics, WeatherNext 3 ensemble, and live regional forecast outputs are populated:

In [ ]:
assert patch_data["precipitation"].shape == (64, 64), "Expected 64x64 storm patch"
assert len(historical_rows) == 36, f"Expected 36 historical rows, got {len(historical_rows)}"
assert len(forecast_rows) == 6, f"Expected 6 forecast rows, got {len(forecast_rows)}"
assert eval_metrics["mae_mm_hr"] >= 0.0, "Expected non-negative MAE"
assert len(wn_data["ensemble_rows"]) > 0, "Expected WeatherNext 3 ensemble rows"
assert len(local_result["forecast_rows"]) == 7, "Expected 72-hour GFS forecast steps"
if abs(float(LOCAL_LATITUDE) - 37.7749) < 1.3 and abs(float(LOCAL_LONGITUDE) - (-122.4194)) < 1.0:
    assert len(local_result["microclimate_rows"]) == 5, "Expected 5 Bay Area microclimate cities"
print("Target state validated: all storm hindcast, ensemble, and live regional forecast checks passed.")

## Cleaning up

To avoid incurring ongoing charges to your Google Cloud account for the resources created in this tutorial, set `CLEANUP_RESOURCES = True` and run the following cell to delete the BigQuery dataset and temporary Cloud Storage staging files (automated test runs clean up resources automatically when `IS_TESTING` is set):

In [ ]:
# fmt: off
CLEANUP_RESOURCES = False  # @param {type: "boolean"}
# fmt: on

if CLEANUP_RESOURCES or os.getenv("IS_TESTING"):
    assert PROJECT_ID and BIGQUERY_DATASET and BUCKET_NAME, "Resource identifiers must be non-empty"
    !bq rm -r -f -d {PROJECT_ID}:{BIGQUERY_DATASET}
    !gcloud storage rm --recursive gs://{BUCKET_NAME}/temp